In [1]:
import pandas as pd
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
# from langchain_community.embeddings import HuggingFaceEmbeddings (deprecated)
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [2]:
books = pd.read_csv("../data/books_cleaned.csv")
books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine..."
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897 Lewis' work on the nature of lov...
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5192,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,On A Train Journey Home To North India After L...,2003.0,2.93,324.0,0.0,Mistaken Identity,9788172235222 On A Train Journey Home To North...
5193,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,24.0,Journey to the East,9788173031014 This book tells the tale of a ma...
5194,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,1568.0,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623 Wisdom to Create a Life of Passi...
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,104.0,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535 This collection of the timeless ...


In [3]:
books["tagged_description"]

0       9780002005883 A NOVEL THAT READERS and critics...
1       9780002261982 A new 'Christie for Christmas' -...
2       9780006178736 A memorable, mesmerizing heroine...
3       9780006280897 Lewis' work on the nature of lov...
4       9780006280934 "In The Problem of Pain, C.S. Le...
                              ...                        
5192    9788172235222 On A Train Journey Home To North...
5193    9788173031014 This book tells the tale of a ma...
5194    9788179921623 Wisdom to Create a Life of Passi...
5195    9788185300535 This collection of the timeless ...
5196    9789027712059 Since the three volume edition o...
Name: tagged_description, Length: 5197, dtype: str

In [4]:
books["tagged_description"].to_csv("../data/tagged_descriptions.txt", sep="\n", index=False, header=False)

In [5]:
loader = TextLoader("../data/tagged_descriptions.txt", encoding="utf-8")
documents = loader.load()

print(documents[0].page_content[:200])

9780002005883 A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a pr


In [6]:
# LangChain updated its rules: chunk_size MUST be > 0.
# To mimic the old behavior (splitting strictly by "\n" without cutting descriptions in half),
# we set chunk_size to a very large number (e.g., 100000).
text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=1,
    chunk_overlap=0
)

book_docs = text_splitter.split_documents(documents)

Created a chunk of size 1168, which is longer than the specified 1
Created a chunk of size 1214, which is longer than the specified 1
Created a chunk of size 373, which is longer than the specified 1
Created a chunk of size 309, which is longer than the specified 1
Created a chunk of size 483, which is longer than the specified 1
Created a chunk of size 482, which is longer than the specified 1
Created a chunk of size 960, which is longer than the specified 1
Created a chunk of size 188, which is longer than the specified 1
Created a chunk of size 843, which is longer than the specified 1
Created a chunk of size 296, which is longer than the specified 1
Created a chunk of size 197, which is longer than the specified 1
Created a chunk of size 881, which is longer than the specified 1
Created a chunk of size 1088, which is longer than the specified 1
Created a chunk of size 1189, which is longer than the specified 1
Created a chunk of size 304, which is longer than the specified 1
Create

In [7]:
print("\n--- First Book Chunk ---")
print(book_docs[0].page_content)


--- First Book Chunk ---
9780002005883 A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance

It worked, and we can just go to the next step, but for me, I don't think this is needed to do this. There should be a better way to do this job without this confusing syntax or method.

---

# Why did the first approach feel strange?

At first, we used this idea:

```python
text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=1,
    chunk_overlap=0
)
```

and it looked very weird. That feeling is correct. The reason is simple:
**we were using the wrong tool for the job.**
Your data is already very clean:

* **1 line = 1 book**
* each line already contains:

  * ISBN
  * description

So the real job is just:

> “split the big text into lines”

But `CharacterTextSplitter` was made for a different purpose.

It was designed more for cases like:

* one very long article
* one long PDF text
* one long document
* then cut it into smaller parts for embeddings / LAG / retrieval

So when we use it for already-clean line-based data, it can feel unnatural.

---

# First, what does `CharacterTextSplitter` actually do?

Think of it like this:

You have a very long text, and you want to cut it into smaller pieces.

It works roughly like this:

1. **split using a separator**

   * for example `"\n"`
2. then **try to combine pieces again**

   * based on `chunk_size`

This second part is the important one.

Many beginners think:

> “If I use `separator="\n"`, then it will just split by newline and stop there.”

But no.

Usually it does:

* split first
* then **merge pieces into chunks**

So it is not just a simple “line splitter”.

That is why it behaved in a strange way for your case.

---

# What is `chunk_size`?

Very simple version:

`chunk_size` means:

> “How big do I want each final chunk to be?”

Usually this size is measured in **characters** by default.

So:

* `chunk_size=1000` means one chunk can contain around 1000 characters
* `chunk_size=500` means one chunk can contain around 500 characters
* `chunk_size=1` means one chunk should contain only **1 character**

And now you can probably already see the problem.

Your first line is something like hundreds of characters long.

So asking for:

```python
chunk_size=1
```

is basically saying:

> “Please make each final chunk only 1 character long.”

But your actual line is much longer than 1 character.

So LangChain complains.

---

# Why did the warning say something like:

> Created a chunk of size 414, which is longer than the specified 1

Because this is what happened:

* LangChain split the text by `\n`
* now it got one line
* that one line was already, for example, **414 characters**
* but you asked for `chunk_size=1`

So LangChain is like:

> “I know you asked for size 1, but this single piece is already size 414. I can’t magically make it smaller without breaking it further in a different way.”

So it keeps that long line, but gives you a warning.

That is why you saw many red warnings.

---

# Then what is `chunk_overlap`?

This is easier.

`chunk_overlap` means:

> “How much of the previous chunk should be repeated in the next chunk?”

Example:

If we have text:

```text
I love machine learning and data science very much.
```

and we split it into chunks, maybe:

* Chunk 1: `"I love machine learning"`
* Chunk 2: `"learning and data science"`
* Chunk 3: `"science very much"`

Notice some words are repeated between chunks.

That repeated part is called **overlap**.

Why do people use overlap?

Because when cutting long text, meaning can get lost at the boundaries.

For example:

* chunk 1 ends with half of an idea
* chunk 2 starts with the rest of the idea

So overlap helps keep context.

---

# Then why was `chunk_overlap=0`?

Because in your case, you do **not** want repetition.

You want:

* line 1 stays line 1
* line 2 stays line 2
* no repeated text

So:

```python
chunk_overlap=0
```

actually makes sense.

That part was not the weird part.

The weird part was mainly:

```python
chunk_size=1
```

---

# Why did `chunk_size=0` give an error?

Because a chunk of size 0 does not make sense.

Think about it.

If I say:

> “Please create chunks of size 0”

that means:

* each chunk should contain nothing (which is the instructor way on this video)

That is logically broken.

A library usually rejects that because:

1. it makes no practical sense
2. it can cause bad internal logic
3. it can even risk infinite loops in some splitting algorithms

So newer LangChain versions are stricter and say:

> `chunk_size` must be greater than 0

That is why `0` throws an error.

So:

* `chunk_size=0` → invalid
* `chunk_size=1` → valid, but terrible for your data

---

# Then why did someone suggest `chunk_size=10000` or `100000`?

That suggestion was trying to “hack around” the problem.

The idea is:

> “If I make `chunk_size` extremely large, then each line will fit comfortably inside it.”

For example:

* one book description line = maybe 300–1500 characters
* `chunk_size=10000` = much bigger than one line

So the hope is:

* each line stays whole
* no warnings

That part is true.

But there is still a hidden problem.

---

# The hidden problem with `chunk_size=10000`

Even if each line fits inside 10000 characters, `CharacterTextSplitter` may still think:

> “Oh, line 1 is small enough.”
> “Line 2 is also small enough.”
> “Let me combine line 1 and line 2 into one bigger chunk, because together they still fit under 10000.”

And that is not what you want.

You want:

* 1 line = 1 chunk
* not 5 lines merged together

So `chunk_size=10000` removes the warnings, but it still does **not** guarantee the behavior you want.

This is why I said the tool itself is the issue.

---

# So what is the real problem?

The real problem is this mismatch:

## Your data shape

Your data is:

* already structured by newline
* each line is already one record

## The tool’s job

`CharacterTextSplitter` is for:

* breaking long text into chunks
* often with chunk-size logic
* often with merging behavior

So you were using a **document chunker** for something that is really just **line splitting**.

That is why it felt weird.

And honestly, your confusion is very normal.

Because from the name:

> `separator="\n"`

it really sounds like it should simply split by newline.

But internally, it is more than that.

---

# What should happen instead?

For your case, the simplest mental model is this:

You have one giant string like:

```text
line1
line2
line3
```

and you only want:

```python
["line1", "line2", "line3"]
```

That is all.

So the correct operation is not:

* “chunk the document”

but:

* “split the text into lines”

That is why plain Python is much more natural here.

---

# The most beginner-friendly way to think about it

Imagine you have a notebook page with 10 book entries.

Each entry is already written on a separate line.

Now compare these two tools:

## Tool A: Scissors for cutting long paragraphs

This tool asks:

* how large should each piece be?
* should pieces overlap?
* should I merge some parts together?

This is like `CharacterTextSplitter`.

## Tool B: Just read line by line

This tool simply says:

* line 1
* line 2
* line 3

This is like:

```python
text.splitlines()
```

For your data, you need **Tool B**, not Tool A.

---

# Why the plain Python way is more “Pythonic”

Because Pythonic code usually means:

* simple
* clear
* direct
* using the most natural built-in method

If your problem is:

> “split text by newline”

then the most natural answer is:

```python
lines = text.splitlines()
```

not a complex text-splitting library.

So the Pythonic idea is:

> use the simplest tool that directly matches the task

---

# Summary of each setting

## `separator="\n"`

Means:

> split using newline

This part matches your data.

But with `CharacterTextSplitter`, splitting by newline is only the **first step**, not the full story.

---

## `chunk_size=0`

Means:

> final chunk size should be 0 characters

This is invalid and the library rejects it.

---

## `chunk_size=1`

Means:

> final chunk size should be 1 character

Technically valid, but absurd for your book descriptions, because each line is much longer than 1 character.

That is why warnings appear.

---

## `chunk_size=10000`

Means:

> final chunk size can be very large

This avoids warnings, but it may merge multiple lines together into one chunk.

So it is still not ideal for “1 line = 1 record”.

---

## `chunk_overlap=0`

Means:

> do not repeat text between chunks

For your case, this is fine.

The main issue was not overlap.

The main issue was using a chunking tool when you only needed line splitting.

---

# The most important lesson

The key lesson is:

> Not every text problem should be solved with a text splitter library.

Sometimes the data is already clean.

Your dataset is one of those cases.

Since every book is already separated by newline, the best idea is simply:

1. split by newline
2. remove empty lines
3. treat each line as one document / one record

That is much cleaner and much easier to understand.

---

# Final beginner-friendly conclusion

The first approach felt weird because:

1. `CharacterTextSplitter` is meant for chunking long text, not simple line-by-line records
2. `chunk_size=1` means “make chunks of only 1 character”, which does not match your long descriptions
3. `chunk_size=0` is invalid because a chunk cannot have size zero
4. `chunk_size=10000` avoids warnings, but can still merge multiple lines together
5. your actual task is much simpler: just split the text by newline

So the strange feeling you had was correct.

The code felt strange because the method itself was not the most natural one for your data.

---

# One simple sentence to remember

**If one line already equals one record, use line splitting, not document chunking.**

In [8]:
import pandas as pd
from langchain_core.documents import Document

# 1. Load the clean dataset from our previous notebook
books = pd.read_csv("../data/books_cleaned.csv")

# 2. The Pythonic Way:
# Convert the Pandas column directly into a list of LangChain Document objects.
# We bypass the messy TextLoader and CharacterTextSplitter entirely!
book_docs = [
    Document(page_content=str(text).strip())
    for text in books["tagged_description"].dropna()
]

# 3. Verify the result
print(f"Successfully created {len(book_docs)} individual book documents.")

# 4. Inspect the very first document to ensure it contains exactly ONE book
print("\n--- First Book Document ---")
print(book_docs[0].page_content)

Successfully created 5197 individual book documents.

--- First Book Document ---
9780002005883 A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s

In [9]:
book_docs[0] # Just to know more what is the result

Document(metadata={}, page_content='9780002005883 A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and 

🏷️ What is metadata inside the Document?
When you inspect a LangChain Document, you will see two main parts:
Document(metadata={}, page_content='...')

page_content (The Core Text): This is the actual raw text (the book description). The AI Embedding Model will only read this part to generate the mathematical vector.

metadata (Data about Data): This is a dictionary used to store extra information or "tags" about the text, such as the author, publish_year, or genre.

Why is ours empty {}?
Because in our code, we only passed the page_content. For a simple Semantic Search, an empty metadata is perfectly fine.

Why is it powerful in production? (Metadata Filtering)
In advanced applications, you might populate it like this:
metadata={"year": 2004, "category": "Fiction"}
When a user searches the Vector Database, you can tell the database: "Only calculate the vector similarity for books where the metadata 'year' is greater than 2000." This allows for blazing-fast hybrid searches (combining exact filters with AI semantic matching).

In [10]:
import torch

print("--- DIAGNOSTIC PYTORCH & GPU ---")
print("Torch version:", torch.__version__)
print("CUDA in torch build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

# well the best thing to do here, is actually having an option to use just the cpu, but iam to lazy to make that, please figure things out by yourself if your device don't have gpu (cuda)

--- DIAGNOSTIK PYTORCH & GPU ---
Torch version: 2.5.1+cu121
CUDA in torch build: 12.1
CUDA available: True
GPU count: 1


In [11]:
# The bge-base-en-v1.5 embedding model converts English sentences and paragraphs into 768-dimensional dense vectors, delivering efficient, ... (https://openrouter.ai/baai/bge-base-en-v1.5)

model_name = "BAAI/bge-base-en-v1.5"
model_kwargs = {'device': 'cuda'} # use the NVIDIA GPU ('cuda')

encode_kwargs = {'normalize_embeddings': True}


print(f"Loading the '{model_name}' model into your NVIDIA GPU...")
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)


persist_dir = "../data/chroma_db"

print("Extracting book meanings and building the Vector Database...")
print("Please wait, the GPU is working...")

db_books = Chroma.from_documents(
    documents=book_docs,
    embedding=embeddings,
    persist_directory=persist_dir
)

print("\nSUCCESS! Vector Database created and safely stored.")


Loading the 'BAAI/bge-base-en-v1.5' model into your NVIDIA GPU...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting book meanings and building the Vector Database...
Please wait, the GPU is working...

SUCCESS! Vector Database created and safely stored.
